In [3]:
import time
import math
import hashlib
import random
from collections import defaultdict

# ==========================================
# [STEP 0] 자체 테스트 로직 (사전 검증)
# ==========================================
def run_internal_tests():
    print("[시스템] 알고리즘 자체 테스트를 시작합니다...")

    # 1. Bloom Filter 테스트
    bf = BloomFilter(m=1000, k=3)
    bf.add("test_user_1")
    bf.add("test_user_2")
    assert bf.check("test_user_1") == True, "BF Test Failed: True Negative"
    assert bf.check("test_user_3") == False, "BF Test Failed: False Positive on empty"

    # 2. Count-Min Sketch 테스트
    cms = CountMinSketch(width=1000, depth=5)
    cms.add("movie_A")
    cms.add("movie_A")
    cms.add("movie_B")
    assert cms.estimate("movie_A") >= 2, "CMS Test Failed: Count underestimation"
    assert cms.estimate("movie_B") >= 1, "CMS Test Failed: Count underestimation"

    print("[시스템] 자체 테스트 완료. 메인 과제 로직을 실행합니다.\n" + "="*50)

# ==========================================
# [STEP 1] 스트리밍 알고리즘 구현
# ==========================================

def get_hash_values(item, k, max_val):
    """
    하나의 아이템에 대해 k개의 독립적인 해시값을 생성 (Kirsch-Mitzenmacher 기법)
    """
    base_hash = hashlib.md5(str(item).encode('utf-8')).hexdigest()
    hash1 = int(base_hash[:16], 16)
    hash2 = int(base_hash[16:], 16)
    return [(hash1 + i * hash2) % max_val for i in range(k)]

class BloomFilter:
    """원소 포함 여부 근사 판정 (Membership Test)"""
    def __init__(self, m, k):
        self.m = m # 비트 배열 크기
        self.k = k # 해시 함수 개수
        self.bit_array = bytearray((m + 7) // 8) # 메모리 최적화를 위한 bytearray 사용

    def add(self, item):
        hashes = get_hash_values(item, self.k, self.m)
        for h in hashes:
            byte_idx = h // 8
            bit_idx = h % 8
            self.bit_array[byte_idx] |= (1 << bit_idx)

    def check(self, item):
        hashes = get_hash_values(item, self.k, self.m)
        for h in hashes:
            byte_idx = h // 8
            bit_idx = h % 8
            if not (self.bit_array[byte_idx] & (1 << bit_idx)):
                return False
        return True

    def get_memory_bytes(self):
        return len(self.bit_array)

class CountMinSketch:
    """항목별 빈도 근사 추정 (Frequency Estimation)"""
    def __init__(self, width, depth):
        self.width = width # 해시 테이블 크기
        self.depth = depth # 해시 함수 개수
        self.table = [[0] * width for _ in range(depth)]

    def add(self, item):
        hashes = get_hash_values(item, self.depth, self.width)
        for i, h in enumerate(hashes):
            self.table[i][h] += 1

    def estimate(self, item):
        hashes = get_hash_values(item, self.depth, self.width)
        return min(self.table[i][h] for i, h in enumerate(hashes))

    def get_memory_bytes(self):
        # 32-bit 정수(4 bytes) 기준 메모리 계산
        return self.width * self.depth * 4

# ==========================================
# [STEP 2 & 3 & 4] 데이터 스트리밍 및 파라미터 비교 실험
# ==========================================

def read_stream(filepath):
    """대용량 데이터를 메모리에 한 번에 올리지 않고 한 줄씩 제너레이터로 반환"""
    try:
        with open(filepath, 'r', encoding='utf-8') as f:
            for line in f:
                yield line.strip().split('::')
    except FileNotFoundError:
        print(f"[오류] {filepath} 파일이 없습니다. 경로를 확인해주세요.")

def experiment_bloom_filter(filepath):
    print(">>> [실험 1] Bloom Filter 파라미터 비교 (메모리 vs 정확도)")

    params = [
        {"m": 100000, "k": 3},
        {"m": 500000, "k": 5},
        {"m": 1000000, "k": 7}
    ]

    for param in params:
        m, k = param["m"], param["k"]
        bf = BloomFilter(m, k)
        ground_truth_set = set()

        start_time = time.time()
        record_count = 0

        for row in read_stream(filepath):
            if len(row) < 2: continue
            user_movie_pair = f"{row[0]}_{row[1]}"
            bf.add(user_movie_pair)
            ground_truth_set.add(user_movie_pair)
            record_count += 1

        processing_time = time.time() - start_time
        if record_count == 0: continue

        false_positives = 0
        test_queries = 100000
        for _ in range(test_queries):
            fake_item = f"user{random.randint(20000, 30000)}_movie{random.randint(20000, 30000)}"
            if fake_item not in ground_truth_set and bf.check(fake_item):
                false_positives += 1

        fpr = (false_positives / test_queries) * 100
        memory_kb = bf.get_memory_bytes() / 1024

        print(f" - 파라미터: m={m:<8}, k={k}")
        print(f"   * 처리 시간: {processing_time:.2f}초 ({record_count/processing_time:.0f} records/sec)")
        print(f"   * 사용 메모리: {memory_kb:.2f} KB")
        print(f"   * 실제 오탐률(FPR): {fpr:.4f} %\n")

def experiment_count_min_sketch(filepath):
    print(">>> [실험 2] Count-Min Sketch 파라미터 비교 (메모리 vs 오차율)")

    params = [
        {"w": 1000, "d": 3},
        {"w": 5000, "d": 5},
        {"w": 10000, "d": 7}
    ]

    for param in params:
        w, d = param["w"], param["d"]
        cms = CountMinSketch(w, d)
        ground_truth_dict = defaultdict(int)

        start_time = time.time()
        record_count = 0

        for row in read_stream(filepath):
            if len(row) < 2: continue
            movie_id = row[1]
            cms.add(movie_id)
            ground_truth_dict[movie_id] += 1
            record_count += 1

        processing_time = time.time() - start_time
        if not ground_truth_dict:
             continue

        total_error = 0
        for movie_id, true_count in ground_truth_dict.items():
            estimated_count = cms.estimate(movie_id)
            total_error += abs(estimated_count - true_count)

        mae = total_error / len(ground_truth_dict)
        memory_kb = cms.get_memory_bytes() / 1024

        print(f" - 파라미터: Width={w:<6}, Depth={d}")
        print(f"   * 처리 시간: {processing_time:.2f}초 ({record_count/processing_time:.0f} records/sec)")
        print(f"   * 사용 메모리: {memory_kb:.2f} KB")
        print(f"   * 평균 절대 오차(MAE): {mae:.2f} 회\n")

if __name__ == "__main__":
    run_internal_tests()
    DATASET_PATH = "/content/sample_data/ratings.dat"

    experiment_bloom_filter(DATASET_PATH)
    experiment_count_min_sketch(DATASET_PATH)

[시스템] 알고리즘 자체 테스트를 시작합니다...
[시스템] 자체 테스트 완료. 메인 과제 로직을 실행합니다.
>>> [실험 1] Bloom Filter 파라미터 비교 (메모리 vs 정확도)
 - 파라미터: m=100000  , k=3
   * 처리 시간: 4.24초 (235623 records/sec)
   * 사용 메모리: 12.21 KB
   * 실제 오탐률(FPR): 100.0000 %

 - 파라미터: m=500000  , k=5
   * 처리 시간: 4.71초 (212233 records/sec)
   * 사용 메모리: 61.04 KB
   * 실제 오탐률(FPR): 99.9680 %

 - 파라미터: m=1000000 , k=7
   * 처리 시간: 5.52초 (181358 records/sec)
   * 사용 메모리: 122.07 KB
   * 실제 오탐률(FPR): 99.3200 %

>>> [실험 2] Count-Min Sketch 파라미터 비교 (메모리 vs 오차율)
 - 파라미터: Width=1000  , Depth=3
   * 처리 시간: 4.34초 (230612 records/sec)
   * 사용 메모리: 11.72 KB
   * 평균 절대 오차(MAE): 352.59 회

 - 파라미터: Width=5000  , Depth=5
   * 처리 시간: 4.10초 (244030 records/sec)
   * 사용 메모리: 97.66 KB
   * 평균 절대 오차(MAE): 1.63 회

 - 파라미터: Width=10000 , Depth=7
   * 처리 시간: 6.11초 (163790 records/sec)
   * 사용 메모리: 273.44 KB
   * 평균 절대 오차(MAE): 0.02 회

